# Demo: Data Cleaning_Week 8-2_20251023

This notebook guides you *step by step* to clean a messy dataset: `messy_retail_data.csv`.

**Reference:** Chapter 7 & 8 in [Python for Data Analysis](https://wesmckinney.com/book/data-cleaning) (3rd edition) by Wes Mckinney.

> Tip: Run one cell at a time. Keep notes of the decisions you make (e.g., how you impute, how you deduplicate).

In [1]:
# Run this to install the sklearn package if needed.
# %pip install -U scikit-learn

In [2]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, PowerTransformer

## 0 Load & Preview
1. Read the CSV file into a DataFrame named `df`  
2. Show the first 5 rows  
3. Show info and basic missing-value counts

In [3]:
csv_path = "messy_retail_data.csv"
df = pd.read_csv(csv_path)

df.head()

,customer_id,purchase_date,channel,age,basket_value,quantity,email,city,note
0,1043,2024-09-23,Mobile,50,20.730943771250146,3 pcs,user9598@mail.test,wenzhou,★ loyal ★
1,1056,2024-09-08,Mobile,30,NaN,NaN,user1357@example.com,WZ,promo#2024
2,1050,2024/09/28,WEB,16,35.78059592617821,6,user8431@example.com,Wenzhou,NaN
3,1015,2024/09/28,WEB,18,60.76571864990895,2,USER5218@EXAMPLE.COM,NaN,VIP
4,1016,NaT,NaN,23,¥7.71,6,user4995@example.com,Shaoxing,


In [4]:
# Quick structure & missing overview
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customer_id    123 non-null    int64 
 1   purchase_date  123 non-null    object
 2   channel        91 non-null     object
 3   age            118 non-null    object
 4   basket_value   113 non-null    object
 5   quantity       116 non-null    object
 6   email          117 non-null    object
 7   city           108 non-null    object
 8   note           106 non-null    object
dtypes: int64(1), object(8)
memory usage: 8.8+ KB


**Task:** What does the output tell us, and how can we interpret it?

## 1 Audit Data Quality (Quick Profiling)
- Check cardinality and sample unique values of key columns
- Look for suspicious values (e.g., negative ages, currency symbols, typos)

**Tasks**
- Show `.nunique()` for each column
- For `channel`, `city`, `quantity`, `age`, show a few unique values

In [5]:
df.nunique()

customer_id       60
purchase_date     70
channel            6
age               59
basket_value     110
quantity           9
email            114
city               7
note               8
dtype: int64

In [6]:
df['city'].nunique()

7

In [7]:
df["channel"].dropna().unique()#
#df["city"].dropna().unique()
#df["quantity"].dropna().unique()[0:10]
#df["age"].dropna().unique()[:10]

array(['Mobile ', 'WEB', 'store', 'mobile', 'Store', 'Web  '],
      dtype=object)

## 2 Handle Missing Values

### 2.1 Identify Missing Values

In [8]:
# Identify the number of missing values for each variable
df.isna().sum()
# An alternative way: pd.isna()

customer_id       0
purchase_date     0
channel          32
age               5
basket_value     10
quantity          7
email             6
city             15
note             17
dtype: int64

**Tasks:**
- What is the outcome of `df.isna()`?
- How to select and display the observations with missing values?

In [9]:
df.isna()

,customer_id,purchase_date,channel,age,basket_value,quantity,email,city,note
0,False,False,False,False,False,False,False,False,False
1,False,False,False,False,True,True,False,False,False
2,False,False,False,False,False,False,False,False,True
3,False,False,False,False,False,False,False,True,False
4,False,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...
118,False,False,False,False,False,False,False,False,False
119,False,False,False,False,False,True,False,False,False
120,False,False,False,False,False,False,False,False,False
121,False,False,False,False,True,False,False,False,False


In [10]:
df[(df['age'].isna()) & (df['city'].isna())]

,customer_id,purchase_date,channel,age,basket_value,quantity,email,city,note
103,1003,2024/09/27,Store,NaN,41.527368516371475,1,user4296@EXAMPLE.com,NaN,RETURNS??


### 2.2 Create Indicator Columns

In [11]:
df["age_isna"] = df["age"].isna().astype(int)
df.head()

,customer_id,purchase_date,channel,age,basket_value,quantity,email,city,note,age_isna
0,1043,2024-09-23,Mobile,50,20.730943771250146,3 pcs,user9598@mail.test,wenzhou,★ loyal ★,0
1,1056,2024-09-08,Mobile,30,NaN,NaN,user1357@example.com,WZ,promo#2024,0
2,1050,2024/09/28,WEB,16,35.78059592617821,6,user8431@example.com,Wenzhou,NaN,0
3,1015,2024/09/28,WEB,18,60.76571864990895,2,USER5218@EXAMPLE.COM,NaN,VIP,0
4,1016,NaT,NaN,23,¥7.71,6,user4995@example.com,Shaoxing,,0


**Your Task:** 
- Create an indicator column `channel_isna`.
- Select observations with missing values in `channel`

In [12]:
df['channel_isna'] = df['channel'].isna().astype(int)
df[df['channel_isna'] == 1]

,customer_id,purchase_date,channel,age,basket_value,quantity,email,city,note,age_isna,channel_isna
4,1016,NaT,NaN,23,¥7.71,6,user4995@example.com,Shaoxing,,0,1
5,1001,09-05-2024,NaN,34,NaN,6,user2535@EXAMPLE.com,Wenzhou,RETURNS??,0,1
7,1047,09-11-2024,NaN,65,42.880269245920424,3,user5064@mail.test,HZ,NaN,0,1
8,1054,2024-09-13,NaN,55,47.559830897627016,6,user4895@ gmail.com,HZ,RETURNS??,0,1
15,1019,2024-09-12,NaN,52,15.057158612075913,4,user3862@ gmail.com,HZ,学生优惠,0,1
16,1048,2024-09-05,NaN,52,¥91.6,3,user2050@mail.test,Sx,NaN,0,1
19,1014,2024/09/21,NaN,28,$26.41,1,user1159@ gmail.com,Shaoxing,promo#2024,0,1
20,1058,2024/09/21,NaN,63,"1,155",4,user1095@ gmail.com,WZ,new-user,0,1
27,1014,2024/09/07,NaN,23,12.067216814279028,7,user6630@example.com,WZ,学生优惠,0,1
28,1018,2024-09-09,NaN,34,9.405287456590457,1,user4974@mail.test,Wenzhou,NaN,0,1


### 2.3 Drop Missing Values

In [13]:
#Drop the rows where all elements are missing.
df.dropna().head()

,customer_id,purchase_date,channel,age,basket_value,quantity,email,city,note,age_isna,channel_isna
0,1043,2024-09-23,Mobile,50,20.730943771250146,3 pcs,user9598@mail.test,wenzhou,★ loyal ★,0,0
9,1031,2024/09/03,Mobile,36,37.1510112161925,2,user9811@EXAMPLE.com,Wenzhou,★ loyal ★,0,0
11,1032,09-27-2024,Mobile,28,23.519954589444623,6,user8151@ gmail.com,WZ,promo#2024,0,0
13,1011,2024-09-15,Mobile,61,78.49803879131842,3,user3325@EXAMPLE.com,Shaoxing,coupon-10%,0,0
17,1034,2024-09-13,Store,68,144.2672832621603,3,user6389@EXAMPLE.com,Sx,★ loyal ★,0,0


**Task**: How can you drop observations where both `age` and `quanity` are missing?
> Reference: For details on how to use `df.dropna()` function, refer to the [official pandas documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.dropna.html).

### 2.4 Mising Value Imputations

#### 2.4.1 Imputation with constant/mean/median/mode

**Think:** What happens when you run the following code?

In [14]:
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df['age_imputed_mean'] = df['age'].fillna(df['age'].mean())
df.head()

,customer_id,purchase_date,channel,age,basket_value,quantity,email,city,note,age_isna,channel_isna,age_imputed_mean
0,1043,2024-09-23,Mobile,50.0,20.730943771250146,3 pcs,user9598@mail.test,wenzhou,★ loyal ★,0,0,50.0
1,1056,2024-09-08,Mobile,30.0,NaN,NaN,user1357@example.com,WZ,promo#2024,0,0,30.0
2,1050,2024/09/28,WEB,16.0,35.78059592617821,6,user8431@example.com,Wenzhou,NaN,0,0,16.0
3,1015,2024/09/28,WEB,18.0,60.76571864990895,2,USER5218@EXAMPLE.COM,NaN,VIP,0,0,18.0
4,1016,NaT,NaN,23.0,¥7.71,6,user4995@example.com,Shaoxing,,0,1,23.0


In [15]:
df[df['age_isna']==1]

,customer_id,purchase_date,channel,age,basket_value,quantity,email,city,note,age_isna,channel_isna,age_imputed_mean
62,1040,2024/09/23,NaN,NaN,53.1343050826726,3,user2168@EXAMPLE.com,Hangzhou,promo#2024,1,1,49.852174
85,1046,2024/09/15,store,NaN,48.97355215454178,5,user4961@example.com,WZ,new-user,1,0,49.852174
101,1055,09-07-2024,WEB,NaN,-30.43367547761865,5,user6485@ gmail.com,Shaoxing,学生优惠,1,0,49.852174
103,1003,2024/09/27,Store,NaN,41.527368516371475,1,user4296@EXAMPLE.com,NaN,RETURNS??,1,0,49.852174
117,1026,2024-09-07,Store,NaN,35.335499926413505,6,user9500@example.com,Hangzhou,VIP,1,0,49.852174


Fill `channel` with `"Unknown"`; **strip** whitespace from text first.

In [16]:
# Convert the 'channel' column to string type and remove leading/trailing spaces
df["channel"] = df["channel"].astype("string").str.strip()
# .str acts as a bridge that allows you to use Python’s string functions (.strip(), .lower(), .replace(), etc.) across an entire column efficiently.

# Replace empty strings ("") with proper missing values (pd.NA)
df["channel"] = df["channel"].replace({"": pd.NA})

# Fill remaining missing values (NaN or NA) with the label "Unknown"
df["channel"] = df["channel"].fillna("Unknown")

**Your Task:** Fill `city` with `"Unknown"`; **strip** whitespace from text first.

#### 2.4.2 Forward / backward fill missing value

In [17]:
df['age_imputed_ffill'] = df['age'].ffill()
df['age_imputed_bfill'] = df['age'].bfill()
df[df['age_isna'] == 1]['age_imputed_ffill'] # show the imputed values

62     74.0
85     40.0
101    22.0
103    56.0
117    40.0
Name: age_imputed_ffill, dtype: float64

> Reference: For details on how to use `df.ffill()` and `df.bfill()` functions, refer to the [official pandas documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.ffill.html).

#### 2.4.3 Interpolate
Fill NaN values using an interpolation method, such as `linear`, `time`,  `index`, `pad`, `nearest`, `krogh`, `from_derivatives`
> Reference: For details on how to use `df.interpolate()` function, refer to the [official pandas documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.interpolate.html)

## Handle Duplicates

In [18]:
df.drop_duplicates()

,customer_id,purchase_date,channel,age,basket_value,quantity,email,city,note,age_isna,channel_isna,age_imputed_mean,age_imputed_ffill,age_imputed_bfill
0,1043,2024-09-23,Mobile,50.0,20.730943771250146,3 pcs,user9598@mail.test,wenzhou,★ loyal ★,0,0,50.0,50.0,50.0
1,1056,2024-09-08,Mobile,30.0,NaN,NaN,user1357@example.com,WZ,promo#2024,0,0,30.0,30.0,30.0
2,1050,2024/09/28,WEB,16.0,35.78059592617821,6,user8431@example.com,Wenzhou,NaN,0,0,16.0,16.0,16.0
3,1015,2024/09/28,WEB,18.0,60.76571864990895,2,USER5218@EXAMPLE.COM,NaN,VIP,0,0,18.0,18.0,18.0
4,1016,NaT,Unknown,23.0,¥7.71,6,user4995@example.com,Shaoxing,,0,1,23.0,23.0,23.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,1050,2024-09-25,WEB,60.0,32.22758965124074,5,user8634@EXAMPLE.com,Wenzhou,学生优惠,0,0,60.0,60.0,60.0
119,1016,09-26-2024,WEB,54.0,$28.09,NaN,USER7823@EXAMPLE.COM,HZ,coupon-10%,0,0,54.0,54.0,54.0
120,1055,2024-09-26,Store,41.0,27.45383823336462,2,user9569@EXAMPLE.com,Hangzhou,★ loyal ★,0,0,41.0,41.0,41.0
121,1053,09-16-2024,mobile,25.0,NaN,1,user7963@example.com,HZ,coupon-10%,0,0,25.0,25.0,25.0


## 3 Duplicates
- Count duplicated **rows** and remove if appropriate
- For duplicated **customer_id**, decide a rule (e.g., keep the **latest** by `purchase_date`)

**Tasks**
1. Report `df.duplicated().sum()` and drop exact duplicates.  
2. For `customer_id`, keep the most recent record by `purchase_date`.


### 3.1 Identify Duplicated Rows 

In [19]:
df.duplicated().sum()

np.int64(3)

> Reference: For details on how to use `df.duplicated()` function, refer to the [official pandas documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html)

**Your Task:** Select and display the duplicated rows

### 3.2 Drop Duplicated Rows

In [20]:
df.drop_duplicates() # Return DataFrame with duplicate rows removed.

,customer_id,purchase_date,channel,age,basket_value,quantity,email,city,note,age_isna,channel_isna,age_imputed_mean,age_imputed_ffill,age_imputed_bfill
0,1043,2024-09-23,Mobile,50.0,20.730943771250146,3 pcs,user9598@mail.test,wenzhou,★ loyal ★,0,0,50.0,50.0,50.0
1,1056,2024-09-08,Mobile,30.0,NaN,NaN,user1357@example.com,WZ,promo#2024,0,0,30.0,30.0,30.0
2,1050,2024/09/28,WEB,16.0,35.78059592617821,6,user8431@example.com,Wenzhou,NaN,0,0,16.0,16.0,16.0
3,1015,2024/09/28,WEB,18.0,60.76571864990895,2,USER5218@EXAMPLE.COM,NaN,VIP,0,0,18.0,18.0,18.0
4,1016,NaT,Unknown,23.0,¥7.71,6,user4995@example.com,Shaoxing,,0,1,23.0,23.0,23.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,1050,2024-09-25,WEB,60.0,32.22758965124074,5,user8634@EXAMPLE.com,Wenzhou,学生优惠,0,0,60.0,60.0,60.0
119,1016,09-26-2024,WEB,54.0,$28.09,NaN,USER7823@EXAMPLE.COM,HZ,coupon-10%,0,0,54.0,54.0,54.0
120,1055,2024-09-26,Store,41.0,27.45383823336462,2,user9569@EXAMPLE.com,Hangzhou,★ loyal ★,0,0,41.0,41.0,41.0
121,1053,09-16-2024,mobile,25.0,NaN,1,user7963@example.com,HZ,coupon-10%,0,0,25.0,25.0,25.0


In [21]:
# Remove duplicates on specific column(s), use subset.
# keep latest per customer_id
df1 = df.drop_duplicates(subset="customer_id", keep="last")
df1.shape

(60, 14)

> Reference: For details on how to use `df.drop_duplicates()` function, refer to the [official pandas documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html)

## 4 Data Type Conversion

### 4.1 `basket_value` → numeric
- Remove commas and currency signs, then `pd.to_numeric(..., errors="coerce")`
- Check negatives and extremes; keep for outlier step

In [22]:
bv = (df["basket_value"]
      .astype("string")
      .str.replace(",", "", regex=False)
      .str.replace("¥", "", regex=False)
      .str.replace("$", "", regex=False)
     )
df["basket_value"] = pd.to_numeric(bv, errors="coerce")

### 4.2 `quantity` → integer
- Convert `"two"` to `2`, `"3 pcs"` to `3`, etc. (hint: regex extract digits, else map words)

In [23]:
q = df["quantity"].astype("string").str.strip()
# extract digits if present; else map common words
digits = q.str.extract(r"(\d+)", expand=False)
word_map = {"one":1, "two":2, "three":3, "four":4, "five":5, "six":6, "seven":7, "eight":8}
as_num = digits.fillna(q.str.lower().map(word_map))
df["quantity"] = pd.to_numeric(as_num, errors="coerce")

### 4.3 `purchase_date` → datetime
- Mixed formats: use `pd.to_datetime(..., errors="coerce", infer_datetime_format=True)`

In [24]:
df["purchase_date"] = pd.to_datetime(df["purchase_date"], errors="coerce", infer_datetime_format=True)

df[["basket_value","quantity","purchase_date"]].head()

C:\Users\wku\AppData\Local\Temp\ipykernel_19268\3864152602.py:1: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df["purchase_date"] = pd.to_datetime(df["purchase_date"], errors="coerce", infer_datetime_format=True)


,basket_value,quantity,purchase_date
0,20.730944,3.0,2024-09-23
1,<NA>,NaN,2024-09-08
2,35.780596,6.0,NaT
3,60.765719,2.0,NaT
4,7.71,6.0,NaT


**Your Tasks:** Clean & convert each column. Create `_before` backups if you’d like to compare.

## 5 Text Cleaning

- Normalize `channel` casing to `Store`, `Mobile`, `Web`, `Unknown`
- Standardize `city` (trim, title-case, map aliases like `WZ`→`Wenzhou`, `HZ`→`Hangzhou`, `Sx`→`Shaoxing`)
- Clean `email` (strip spaces, lower-case), then check for rudimentary validity (contains `'@'` and a domain)
- Create a cleaned `note_clean` removing special characters and collapsing whitespace

In [25]:
# Channel normalization
df["channel"] = df["channel"].str.strip().str.lower()
df["channel"] = df["channel"].replace({
    "store":"store","mobile ":"mobile","mobile":"mobile","web":"web","web  ":"web"
})
df["channel"] = df["channel"].replace({"unknown":"unknown", None:"unknown"})
df["channel"] = df["channel"].map({"store":"Store","mobile":"Mobile","web":"Web","unknown":"Unknown"}).fillna("Unknown")
df["channel"] = df["channel"].astype("category")

# City normalization
df["city"] = df["city"].str.strip().str.title()
city_map = {"Wz":"Wenzhou","Hz":"Hangzhou","Sx":"Shaoxing"}
df["city"] = df["city"].replace({"Wz":"Wenzhou","Hz":"Hangzhou","Sx":"Shaoxing","Unknown":"Unknown"})
df["city"] = df["city"].replace({"Wz":"Wenzhou","Hz ":"Hangzhou","Hz":"Hangzhou","Wenzhou":"Wenzhou"})
df["city"] = df["city"].replace({"Sx":"Shaoxing","Wz ":"Wenzhou"," Hz ":"Hangzhou"})
df["city"] = df["city"].fillna("Unknown").astype("category")

# Email cleaning & basic validity
df["email"] = df["email"].astype("string").str.strip().str.lower()
df["email_valid"] = df["email"].str.contains(r"^[^\s@]+@[^\s@]+\.[^\s@]+$", na=False)

# Note cleaning
df["note_clean"] = (df["note"].astype("string")
                    .str.strip()
                    .str.replace(r"[^A-Za-z0-9\u4e00-\u9fff\s\-]+", "", regex=True)
                    .str.replace(r"\s+", " ", regex=True)
                    .str.lower()
                   )
df[["channel","city","email","email_valid","note","note_clean"]].head()

,channel,city,email,email_valid,note,note_clean
0,Mobile,Wenzhou,user9598@mail.test,True,★ loyal ★,loyal
1,Mobile,Wenzhou,user1357@example.com,True,promo#2024,promo2024
2,Web,Wenzhou,user8431@example.com,True,NaN,<NA>
3,Web,Unknown,user5218@example.com,True,VIP,vip
4,Unknown,Shaoxing,user4995@example.com,True,,


## 6 Outliers (on `basket_value`)
- Compute IQR bounds and Z-scores
- Create flags: `basket_iqr_outlier`, `basket_z_outlier`
- Apply **capping** (winsorization) to bring values within IQR bounds (keep original in `basket_value_raw`)

In [26]:
x = df["basket_value"]
Q1, Q3 = x.quantile(0.25), x.quantile(0.75)
IQR = Q3 - Q1
low, high = Q1 - 1.5*IQR, Q3 + 1.5*IQR

z = np.abs(stats.zscore(x, nan_policy="omit"))
z = pd.Series(z, index=x.index)

df["basket_iqr_outlier"] = (x < low) | (x > high)
df["basket_z_outlier"] = z > 3

df["basket_value_raw"] = df["basket_value"]
df["basket_value_capped"] = x.clip(lower=low, upper=high)

df[["basket_value_raw","basket_value_capped","basket_iqr_outlier","basket_z_outlier"]].head(10)

,basket_value_raw,basket_value_capped,basket_iqr_outlier,basket_z_outlier
0,20.730944,20.730944,False,False
1,<NA>,<NA>,<NA>,False
2,35.780596,35.780596,False,False
3,60.765719,60.765719,False,False
4,7.71,7.71,False,False
5,<NA>,<NA>,<NA>,False
6,21.21,21.21,False,False
7,42.880269,42.880269,False,False
8,47.559831,47.559831,False,False
9,37.151011,37.151011,False,False


**Your Tasks:** Understand each line of codes above

## 7 Normalization & Scaling (post-cleaning)
Choose numeric columns (e.g., `age`, `basket_value_capped`, `quantity`) and demonstrate:
- **MinMaxScaler**
- **StandardScaler** (Z-score)
- **RobustScaler**

Create a small `scaled` DataFrame with each version.

In [27]:
num_cols = ["age", "basket_value_capped", "quantity"]
X = df[num_cols].dropna().to_numpy()

scaled = pd.DataFrame({
    "age_minmax": MinMaxScaler().fit_transform(X)[:,0],
    "age_z": StandardScaler().fit_transform(X)[:,0],
    "age_robust": RobustScaler().fit_transform(X)[:,0],
    "age_yeoj": PowerTransformer(method="yeo-johnson").fit_transform(X)[:,0],
    "basket_minmax": MinMaxScaler().fit_transform(X)[:,1],
    "basket_z": StandardScaler().fit_transform(X)[:,1],
    "basket_robust": RobustScaler().fit_transform(X)[:,1],
})
scaled.head()

,age_minmax,age_z,age_robust,age_yeoj,basket_minmax,basket_z,basket_robust
0,0.337748,-0.070134,-0.060150,0.014046,0.389184,-0.661738,-0.294920
1,0.112583,-1.448498,-1.082707,-1.596300,0.465418,-0.304056,-0.004968
2,0.125828,-1.367418,-1.022556,-1.481281,0.591981,0.289760,0.476406
3,0.158940,-1.164718,-0.872180,-1.210253,0.323226,-0.971205,-0.545787
4,0.437086,0.537967,0.390977,0.594207,0.501382,-0.135320,0.131817


## 8 Join & Reshape (quick demo)
- Create a small mapping (city → region) and **merge** it  
- Build a simple monthly pivot from `purchase_date` and `basket_value_capped`

1. Map `city` to a region (e.g., Wenzhou/Shaoxing → “Zhejiang South”, Hangzhou → “Zhejiang North”, Unknown → “Unknown”).  
2. Create a pivot table: index=`city`, columns=`month`, values=`basket_value_capped` (mean).


In [28]:
map_df = pd.DataFrame({
    "city": ["Wenzhou","Shaoxing","Hangzhou","Unknown"],
    "region": ["Zhejiang South","Zhejiang South","Zhejiang North","Unknown"]
})
df = df.merge(map_df, on="city", how="left")

df["month"] = df["purchase_date"].dt.to_period("M").astype("string")
pivot = df.pivot_table(index="city", columns="month", values="basket_value_capped", aggfunc="mean")
pivot.head()

month,2024-09
city,
Hangzhou,38.589299
Shaoxing,59.686063
Unknown,81.469216
Wenzhou,41.227022


## 9 Save Cleaned Dataset
Save a cleaned, analysis-ready file with a clear name.

In [29]:
clean_path = "messy_retail_data_cleaned.csv"
df.to_csv(clean_path, index=False)
clean_path

'messy_retail_data_cleaned.csv'


### ✅ Wrap-Up
In this workbook, you:
- Audited and fixed **missing values**, **types**, and **text noise**
- Standardized **categoricals**
- Detected and **capped outliers**
- Demonstrated **scaling**
- Practiced **merge** and **pivot**

> Deliverables suggestion: export the cleaned CSV and write a short summary (5–8 bullets) documenting your decisions.